# 01 — Zero-shot PIT audit (TimesFM 2.5, Phase 1)

Thin driver. All logic lives in the `tsfm_cal` package so it stays diffable and
reusable by later notebooks. Runs on **Kaggle T4**.

Canonical run: **log returns**, `force_flip_invariance=False` (asymmetry
preserved). See `HANDOFF_1_JUNE_22_2026.md`.

## 1. Install + clone (then **RESTART KERNEL** before importing timesfm)

In [ ]:
!pip install "git+https://github.com/google-research/timesfm.git#egg=timesfm[torch]" -q
!pip install yfinance -q
# Replace <user> with your GitHub user once the repo is pushed:
!pip install -e "git+https://github.com/<user>/tsfm-thesis.git#egg=tsfm_cal" -q
# >>> Runtime -> Restart kernel, then continue from cell 2. <<<

## 2. Imports + load & compile the model (flags explicit, flip OFF)

In [ ]:
import torch, timesfm
from tsfm_cal import config, data, pit, eval, plot, io_utils

torch.set_float32_matmul_precision("high")
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
model.compile(timesfm.ForecastConfig(**config.FORECAST_FLAGS))
print("Model loaded. Flags:", config.FORECAST_FLAGS)

## 2b. SMOKE TEST (run first) — 1 asset, 100 steps, ~1 min

Validates install + model + PIT + save path end-to-end before the full run.
Does NOT write into a canonical run folder. Expect a finite KS and a populated
`pit`/`quantiles` array. If this works, proceed to cell 3.

In [ ]:
import numpy as np
_dates, _r = data.download_returns("SPY", end="2010-01-01")          # short window
_r = _r[:config.CONTEXT + 100]                                       # ~100 forecast steps
_res = pit.compute_pit_for_asset(model, _r, label="SMOKE", progress_every=0)
_ks, _p = eval.ks_uniform(_res["pit"])
assert _res["quantiles"].shape[1] == 9 and _res["pit"].size > 50, "smoke shapes wrong"
assert eval.quantile_crossing_rate(_res["quantiles"]) == 0.0, "quantiles crossing!"
print(f"SMOKE OK  n={_res['pit'].size}  KS={_ks:.4f}  p={_p:.3f}  "
      f"q-range[{_res['quantiles'].min():.4f}, {_res['quantiles'].max():.4f}]")

## 3. Create the run folder + record the exact config

In [ ]:
rid = config.run_id(flip=False)
run_dir = io_utils.zeroshot_run_dir(rid)
io_utils.save_json(run_dir / "config.json", {
    "run_id": rid,
    "flags": config.FORECAST_FLAGS,
    "returns_type": config.RETURNS_TYPE,
    "start": config.START, "end": config.END,
    "context": config.CONTEXT, "horizon": config.HORIZON,
    "tickers": config.TICKERS,
})
print("Run dir:", run_dir)

## 4. Run all 7 assets (saves each asset's npz **inside** the loop)

In [ ]:
results = pit.run_all_assets(model, run_dir)
print("Done:", list(results.keys()))

## 5. Diagnostics table -> pit_summary.csv + ks_table.json

In [ ]:
rows = eval.summarize_run(run_dir, from_disk=True)
io_utils.save_table_csv(run_dir / "tables" / "pit_summary.csv", rows)
io_utils.save_json(run_dir / "tables" / "ks_table.json",
                   {r["asset"]: {"ks": r["ks"], "p": r["ks_p"]} for r in rows})
for r in rows:
    print(f'{r["asset"]:<18} KS={r["ks"]:.4f} p={r["ks_p"]:.2e} '
          f'cov80={r["cov80"]:.3f} tail_asym={r["tail_asym"]:+.3f}')

## 6. Figures: 2x4 PIT grid + KS-vs-kurtosis

In [ ]:
pits = {r["asset"]: io_utils.load_pit_npz(run_dir, r["asset"])["pit"] for r in rows}
ks   = {r["asset"]: r["ks"] for r in rows}
plot.pit_grid(pits, ks, save_path=run_dir / "figures" / "pit_grid.png")
plot.ks_vs_kurtosis(rows, save_path=run_dir / "figures" / "ks_vs_kurtosis.png")
print("Figures saved.")

## 7. Conditional (regime-stratified) KS for SPY — aggregate can hide collapse

In [ ]:
spy = io_utils.load_pit_npz(run_dir, "Equities (SPY)")
for s in eval.conditional_ks(spy["pit"], spy["realized"], n_strata=5):
    print(s)

## 8. SPY symmetrization comparison (flip=True) — quantify the old confound

Re-compiles with `force_flip_invariance=True` and re-runs SPY only, into a
separate run_id, so flip=False vs flip=True can be compared directly.

In [ ]:
model.compile(timesfm.ForecastConfig(**config.FORECAST_FLAGS_FLIP))
rid_flip = config.run_id(flip=True)
run_dir_flip = io_utils.zeroshot_run_dir(rid_flip)
io_utils.save_json(run_dir_flip / "config.json",
                   {"run_id": rid_flip, "flags": config.FORECAST_FLAGS_FLIP,
                    "returns_type": config.RETURNS_TYPE, "note": "SPY-only flip comparison"})

_dates, spy_rets = data.download_returns("SPY")
res = pit.compute_pit_for_asset(model, spy_rets, label="SPY-flip")
io_utils.save_pit_npz(run_dir_flip, "Equities (SPY)",
                      pit=res["pit"], pit_interp=res["pit_interp"],
                      quantiles=res["quantiles"], realized=res["realized"])
ks_f, p_f = eval.ks_uniform(res["pit"])
ks_n, p_n = eval.ks_uniform(spy["pit"])
print(f"SPY  flip=False: KS={ks_n:.4f}  |  flip=True: KS={ks_f:.4f}")
print("tail (flip=False):", eval.tail_mass(spy["pit"]))
print("tail (flip=True): ", eval.tail_mass(res["pit"]))

## 9. Artifact listing (confirm everything persisted)

In [ ]:
import os
for root, _dirs, files in os.walk(io_utils.base_dir() / "zeroshot"):
    for fn in sorted(files):
        print(os.path.join(root, fn))